# HW4 Part 5: Monitoring & Drift Detection

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib

from matplotlib.patches import Circle, FancyBboxPatch
from scipy.stats import ks_2samp
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

np.random.seed(42)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print('All imports successful.')

## Step 1: Load Reference Data and Trained Model

In [ ]:
# Load training data (reference distribution) and trained pipeline
ref_data = pd.read_csv('ml_ready_df.csv')
model    = joblib.load('model/model.pkl')

TARGET = 'is_positive_review'
FEATURE_COLS = [c for c in ref_data.columns if c != TARGET]

print(f'Reference data: {len(ref_data):,} records')
print(f'Positive review rate: {ref_data[TARGET].mean():.1%}')
print(f'\nFeature statistics (selected):')
print(ref_data[['delivery_days', 'freight_value', 'price', 'delivery_vs_estimated', 'freight_ratio']].describe().round(2))

## Step 2: Simulate 6 Months of Production Data

- **Months 1-3:** Stable — small random noise (+-5%) only
- **Months 4-6:** Progressive drift in `delivery_days`, `freight_value`, `product_category_name_english` + 5% label flips

---

In [ ]:
NUMERIC_COLS = [
    'delivery_days', 'delivery_vs_estimated', 'price', 'freight_value',
    'seller_score', 'num_previous_sales', 'cust_reviews', 'freight_ratio',
    'num_previous_reviews', 'num_items', 'same_state', 'is_repeat_customer',
    'delivery_missing'
]

def simulate_month(ref_data, month, n_samples=3000):
    """Simulate one month of production data with optional drift."""
    sample = ref_data.sample(n=n_samples, replace=True, random_state=42 + month).copy()
    sample = sample.reset_index(drop=True)

    # +-5% noise for ALL months
    for col in NUMERIC_COLS:
        noise = np.random.normal(1.0, 0.05, len(sample))
        sample[col] = sample[col] * noise

    # Progressive drift for months 4-6
    if month >= 4:
        drift_level = month - 3  # 1, 2, 3

        # delivery_days: +2 days per drift level
        sample['delivery_days'] = sample['delivery_days'] + (2 * drift_level)

        # freight_value: +15% / +30% / +50%
        freight_increase = {1: 1.15, 2: 1.30, 3: 1.50}[drift_level]
        sample['freight_value'] = sample['freight_value'] * freight_increase

        # freight_ratio: recompute after freight_value change
        sample['freight_ratio'] = sample['freight_value'] / (sample['price'] + 1e-9)

        # product_category: boost electronics share by 10%/15%/20%
        boost = {1: 0.10, 2: 0.15, 3: 0.20}[drift_level]
        n_to_flip = int(len(sample) * boost)
        non_elec_idx = sample[sample['product_category_name_english'] != 'electronics'].index
        flip_idx = np.random.choice(non_elec_idx, size=min(n_to_flip, len(non_elec_idx)), replace=False)
        sample.loc[flip_idx, 'product_category_name_english'] = 'electronics'

        # Concept drift: flip 5% of positive labels to negative
        positive_idx = sample[sample[TARGET] == 1].index
        n_flip = int(len(positive_idx) * 0.05)
        if n_flip > 0 and len(positive_idx) > 0:
            flip_label_idx = np.random.choice(positive_idx, size=min(n_flip, len(positive_idx)), replace=False)
            sample.loc[flip_label_idx, TARGET] = 0

    sample['month'] = month
    return sample


# Generate all 6 months
production = pd.concat(
    [simulate_month(ref_data, m) for m in range(1, 7)],
    ignore_index=True
)

print(f'Total production records: {len(production):,}')
print(f'\nRecords per month: {production["month"].value_counts().sort_index().to_dict()}')
print(f'\nPositive review rate per month:')
for m in range(1, 7):
    rate = production[production['month'] == m][TARGET].mean()
    marker = ' <- drift starts' if m == 4 else ''
    print(f'  Month {m}: {rate:.3f}{marker}')

In [ ]:
# Verify drift is visible in feature means
print('Feature means per month:')
print('=' * 60)
summary = production.groupby('month')[['delivery_days', 'freight_value', 'price', 'delivery_vs_estimated', 'freight_ratio']].mean().round(2)
print(summary.to_string())

## Step 3: Calculate PSI (Population Stability Index)

> PSI = Sum of (Actual% - Expected%) x ln(Actual% / Expected%)

We bin both distributions, compare the proportions in each bin, and sum up the differences. The log term penalizes proportional changes more than absolute ones.

---

In [ ]:
def calculate_psi(reference, current, bins=10):
    """
    Calculate Population Stability Index.

    PSI < 0.1        -> No significant drift (stable)
    0.1 <= PSI < 0.2 -> Moderate drift (investigate)
    PSI >= 0.2       -> Significant drift (action required)
    """
    breakpoints = np.linspace(
        min(reference.min(), current.min()),
        max(reference.max(), current.max()),
        bins + 1
    )
    ref_counts = np.histogram(reference, bins=breakpoints)[0]
    cur_counts = np.histogram(current,   bins=breakpoints)[0]

    # Proportions — small constant avoids division by zero
    ref_pct = ref_counts / len(reference) + 1e-4
    cur_pct = cur_counts / len(current)   + 1e-4

    psi = np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct))
    return round(psi, 4)


def interpret_psi(val):
    if val < 0.1:   return 'Stable'
    elif val < 0.2: return 'Moderate'
    else:           return 'Significant'


# Features to monitor with PSI
PSI_FEATURES = ['delivery_days', 'freight_value', 'price', 'delivery_vs_estimated', 'freight_ratio']

# Calculate PSI for each feature x month
psi_results = {}
for feature in PSI_FEATURES:
    psi_results[feature] = {}
    for month in range(1, 7):
        month_data = production[production['month'] == month][feature]
        psi_results[feature][month] = calculate_psi(ref_data[feature], month_data)

# Display
psi_df = pd.DataFrame(psi_results).T
psi_df.columns = [f'Month {m}' for m in range(1, 7)]

print('PSI Results (Reference vs Each Month)')
print('=' * 70)
print(psi_df.round(4).to_string())

### Interpret the results:
Look at `delivery_days` and `freight_value` — months 1-3 are well under 0.1 (stable). Month 4 starts creeping up as drift is introduced. Notice `price` stays stable throughout — we did not drift it, and the PSI confirms it.

## Step 4: KS Test (Kolmogorov-Smirnov)

In [ ]:
# KS test for each feature x month
print('KS Test Results')
print('=' * 70)

for feature in PSI_FEATURES:
    print(f'\n{feature}:')
    for month in range(1, 7):
        month_data = production[production['month'] == month][feature]
        stat, pval = ks_2samp(ref_data[feature], month_data)
        sig = '*** SIGNIFICANT' if pval < 0.05 else ''
        print(f'  Month {month}: KS={stat:.4f}, p={pval:.4f} {sig}')

### PSI vs KS:
PSI gives a severity score (0.15 = moderate). KS gives a yes/no answer (p < 0.05 = significant). Both tools tell the same story. Use PSI for dashboards and tracking over time, KS for statistical confirmation.

## Step 5: Performance Monitoring

In [ ]:
# Generate predictions on all production data
production['pred']      = model.predict(production[FEATURE_COLS])
production['pred_prob'] = model.predict_proba(production[FEATURE_COLS])[:, 1]

# Monthly metrics
monthly_metrics = []
for month in range(1, 7):
    mask   = production['month'] == month
    y_true = production.loc[mask, TARGET]
    y_pred = production.loc[mask, 'pred']
    y_prob = production.loc[mask, 'pred_prob']

    monthly_metrics.append({
        'Month':    month,
        'Accuracy': accuracy_score(y_true, y_pred),
        'F1':       f1_score(y_true, y_pred),
        'AUC':      roc_auc_score(y_true, y_prob),
    })

metrics_df = pd.DataFrame(monthly_metrics).set_index('Month')

print('Monthly Performance Metrics')
print('=' * 50)
print(metrics_df.round(4).to_string())

# Performance drop
print(f'\nPerformance Drop (Month 1 -> Month 6):')
for col in ['Accuracy', 'F1', 'AUC']:
    m1   = metrics_df.loc[1, col]
    m6   = metrics_df.loc[6, col]
    drop = m1 - m6
    pct  = (drop / m1) * 100
    print(f'  {col}: {m1:.4f} -> {m6:.4f}  (down {drop:.4f}, {pct:.1f}%)')

### Connect drift to performance:
Feature drift is visible in Month 4 (PSI starts rising for `delivery_days` and `freight_value`). Performance degradation follows in Months 5-6. Feature drift is the **early warning**. Performance drop is the **impact**. You need both signals.

## Step 6: 5-Panel Monitoring Dashboard

In [ ]:
fig = plt.figure(figsize=(24, 14))
fig.suptitle('Olist Model Health Dashboard — 6-Month Production Monitoring',
             fontsize=18, fontweight='bold', y=0.98)

gs = gridspec.GridSpec(2, 6, hspace=0.35, wspace=0.5,
                       top=0.92, bottom=0.05, left=0.05, right=0.95)

# Panel 1: PSI Heatmap
ax1 = fig.add_subplot(gs[0, 0:2])
psi_matrix = psi_df.values.astype(float)
cmap = mcolors.LinearSegmentedColormap.from_list('drift', ['#2ecc71', '#f1c40f', '#e74c3c'], N=256)

sns.heatmap(
    psi_matrix, ax=ax1, annot=True, fmt='.3f', cmap=cmap,
    xticklabels=[f'M{m}' for m in range(1, 7)],
    yticklabels=PSI_FEATURES,
    vmin=0, vmax=0.3,
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'PSI', 'shrink': 0.8}
)
ax1.set_title('Feature Drift (PSI)', fontsize=13, fontweight='bold', pad=10)
ax1.set_xlabel('Month')
ax1.set_ylabel('')

# Panel 2: Performance Trend
ax2 = fig.add_subplot(gs[0, 2:4])
months = range(1, 7)
ax2.plot(months, metrics_df['F1'],       'o-', color='#2e86ab', lw=2.5, ms=9, label='F1 Score',  zorder=3)
ax2.plot(months, metrics_df['AUC'],      's-', color='#ed7d31', lw=2.5, ms=9, label='AUC',       zorder=3)
ax2.plot(months, metrics_df['Accuracy'], '^-', color='#70ad47', lw=2,   ms=8, label='Accuracy',  alpha=0.7, zorder=2)
ax2.axhline(y=0.70, color='red', ls='--', lw=1.5, alpha=0.7, label='Alert Threshold (F1=0.70)')
ax2.fill_between(months, 0, 0.70, alpha=0.05, color='red')
ax2.set_xlabel('Month', fontsize=11)
ax2.set_ylabel('Score', fontsize=11)
ax2.set_title('Performance Trend', fontsize=13, fontweight='bold', pad=10)
ax2.set_xticks(range(1, 7))
ax2.set_ylim(0.5, 1.0)
ax2.legend(loc='lower left', fontsize=9)
ax2.grid(True, alpha=0.3)
for m, val in zip(months, metrics_df['F1']):
    ax2.annotate(f'{val:.3f}', (m, val), textcoords='offset points',
                 xytext=(0, 8), fontsize=8, color='#2e86ab', ha='center')

# Panel 3: Distribution Comparison
ax3 = fig.add_subplot(gs[0, 4:6])
ref_col = ref_data['delivery_days']
m6_col  = production[production['month'] == 6]['delivery_days']
ax3.hist(ref_col, bins=40, alpha=0.5, color='steelblue', label='Reference',
         density=True, edgecolor='white', linewidth=0.5)
ax3.hist(m6_col,  bins=40, alpha=0.5, color='coral',     label='Month 6 (Drifted)',
         density=True, edgecolor='white', linewidth=0.5)
ax3.axvline(ref_col.mean(), color='steelblue', ls='--', lw=2, alpha=0.8)
ax3.axvline(m6_col.mean(),  color='coral',     ls='--', lw=2, alpha=0.8)
ax3.set_title('delivery_days: Reference vs Month 6', fontsize=13, fontweight='bold', pad=10)
ax3.set_xlabel('Delivery Days', fontsize=11)
ax3.set_ylabel('Density', fontsize=11)
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.2)

# Panel 4: Model Health Score (traffic light)
ax4 = fig.add_subplot(gs[1, 0:3])
ax4.set_xlim(0, 10)
ax4.set_ylim(0, 10)
ax4.axis('off')
ax4.set_title('Model Health Score', fontsize=13, fontweight='bold', pad=10)

avg_psi_m6   = np.mean([psi_results[f][6] for f in PSI_FEATURES])
psi_health   = max(0, 1 - avg_psi_m6 / 0.3)
f1_baseline  = metrics_df.loc[1, 'F1']
f1_current   = metrics_df.loc[6, 'F1']
f1_health    = min(1.0, f1_current / f1_baseline)
health_score = 0.5 * psi_health + 0.5 * f1_health

if health_score >= 0.8:
    status_text, status_color = 'HEALTHY',  '#27ae60'
elif health_score >= 0.6:
    status_text, status_color = 'WARNING',  '#f39c12'
else:
    status_text, status_color = 'CRITICAL', '#c0392b'

traffic_bg = FancyBboxPatch((0.5, 1), 3.5, 8, boxstyle='round,pad=0.3',
                             facecolor='#2d2d2d', edgecolor='#1a1a1a', linewidth=2)
ax4.add_patch(traffic_bg)
for cy, color, active in [
    (7.5, '#2ecc71', health_score >= 0.8),
    (5.0, '#f1c40f', 0.6 <= health_score < 0.8),
    (2.5, '#e74c3c', health_score < 0.6)
]:
    ax4.add_patch(Circle((2.25, cy), 1.0,
                          facecolor=color if active else '#444444',
                          edgecolor='#333333', linewidth=1.5,
                          alpha=1.0 if active else 0.3))

ax4.text(6.5, 7.5, f'{health_score:.0%}', fontsize=48, fontweight='bold',
         color=status_color, ha='center', va='center')
ax4.text(6.5, 5.5, status_text, fontsize=22, fontweight='bold',
         color=status_color, ha='center', va='center')
ax4.text(5, 3.5, f'Drift Score: {psi_health:.0%}',       fontsize=12, color='#666666', ha='left', va='center')
ax4.text(5, 2.5, f'Performance Score: {f1_health:.0%}',   fontsize=12, color='#666666', ha='left', va='center')
ax4.text(5, 1.5, f'(PSI avg: {avg_psi_m6:.3f} | F1: {f1_current:.3f})', fontsize=10, color='#999999', ha='left', va='center')

# Panel 5: Alert Summary
ax5 = fig.add_subplot(gs[1, 3:6])
ax5.axis('off')
ax5.set_title('Alert Summary', fontsize=13, fontweight='bold', pad=10)

alerts = []
for feature in PSI_FEATURES:
    for month in range(1, 7):
        psi_val  = psi_results[feature][month]
        severity = interpret_psi(psi_val)
        if severity != 'Stable':
            action = 'Investigate' if severity == 'Moderate' else 'ACTION REQUIRED'
            alerts.append([feature, f'M{month}', f'{psi_val:.3f}', severity, action])

if alerts:
    col_labels = ['Feature', 'Month', 'PSI', 'Severity', 'Action']
    table = ax5.table(
        cellText=alerts, colLabels=col_labels,
        cellLoc='center', loc='upper center',
        colWidths=[0.22, 0.10, 0.12, 0.18, 0.28]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.4)
    for j in range(len(col_labels)):
        table[0, j].set_facecolor('#1f3a5f')
        table[0, j].set_text_props(color='white', fontweight='bold')
    for i, alert in enumerate(alerts, 1):
        color = '#fff3cd' if alert[3] == 'Moderate' else '#f8d7da'
        for j in range(len(col_labels)):
            table[i, j].set_facecolor(color)
else:
    ax5.text(0.5, 0.5, 'No alerts — all features stable', ha='center', va='center',
             fontsize=14, color='green')

plt.savefig('monitoring_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nThis 5-panel dashboard answers: Is the model healthy? in 5 seconds.')
print('  Panel 1 (Heatmap):      WHERE is drift occurring?')
print('  Panel 2 (Trend):        HOW is performance changing?')
print('  Panel 3 (Distribution): WHAT does the drift look like?')
print('  Panel 4 (Health Score): Overall status at a glance')
print('  Panel 5 (Alerts):       WHAT needs attention?')

## Step 7: Retraining Recommendation

The monitoring results show gradual but consistent model degradation beginning in Month 4. Among the five monitored features, `delivery_days` crossed the moderate-drift threshold (PSI = 0.124) in Month 6, while `freight_value` and other features remained stable throughout. No features crossed the 0.2 significant-drift threshold during the observation period.

Despite the relatively modest feature drift, model performance declined steadily during the drift period: F1 dropped from 0.883 in Month 1 to 0.827 in Month 6 (absolute drop: 0.056, 6.3% decrease), and ROC-AUC fell from 0.843 to 0.784 (absolute drop: 0.059, 7.0% decrease). Both metrics remained above the 0.70 acceptable threshold throughout.

**Should the model be retrained?** Not immediately, but proactive monitoring should continue.

**Recommended triggers:**
- **PSI threshold:** Trigger a retraining review when any feature PSI exceeds 0.2 for two consecutive months
- **Performance threshold:** Trigger if F1 drops below 0.82 — a conservative buffer that would have activated at Month 4 in this simulation
- **Scheduled cadence:** Retrain quarterly as a precaution even without alerts

**Data strategy:** Use a rolling 12-month window of production data (with verified labels), weighted toward recent months to reflect the shifting `delivery_days` distribution and higher `freight_value` levels observed in Months 4-6. The retrained model should also reflect the increased electronics share in `product_category_name_english`.